# Librerias necesarias: 
para relaizar el web scraping; al parecer la pagina del gobierno es una pagina estatica entonces usaremos request y la librerias beatifoulsoup para hacer el arbol html de la pagina(parsear el documento)

In [7]:
import pandas as pd 
import numpy as np
from bs4 import BeautifulSoup
import requests
from io import BytesIO
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
import openpyxl
import re
import unicodedata


# extraccion datos
con el link de la pagina accedemos a esta y extraemos su htmlatravez de un request

In [6]:
url="https://www.supersolidaria.gov.co/es/entidad/cooperativas-de-ahorro-y-credito"
response = requests.get(url, verify=False)
soup = BeautifulSoup(response.text, 'html.parser')

/home/lia/Descargas/Anaconda/envs/camel-model/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


# organizacion de los links 
Ahora lo que se procede a hacer es que organizamos los links de los archivos a traves para posteriormente descargarlos

In [7]:
etiqueta="a"
documentos = soup.find_all(etiqueta, href=lambda href: href and (href.endswith(".xlsx") or href.endswith(".xls") ) )
links_descarga=[documentos[i]["href"] for i in range(len(documentos))]

In [10]:
def nombre(ruta):
    nombre_archivo = ruta.rsplit("/", 1)[-1] if "/" in ruta else "archivo_descargado.xlsx"
    return nombre_archivo.strip("\"'")

def Filtrar_datos(list_links):
    Links_descarga_modif = list(map(lambda ruta: nombre(ruta).split("_"), list_links))
    indices_a_conservar = [i for i in range(len(Links_descarga_modif)) 
                        if ("estados" in Links_descarga_modif[i] and "financieros" in Links_descarga_modif[i]) or 
                        ("financiera" in Links_descarga_modif[i] and "puc" in Links_descarga_modif[i]) or 
                        ("financiera" in Links_descarga_modif[i] and "f" in Links_descarga_modif[i]) or 
                        ("Estados" in Links_descarga_modif[i] and "Financieros" in Links_descarga_modif[i]) or 
                        ("estados" in Links_descarga_modif[i] and "%20financieros" in Links_descarga_modif[i]) or
                        ("Estados" in Links_descarga_modif[i] and "%20Financieros" in Links_descarga_modif[i]) or
                        ("20231117estados" in Links_descarga_modif[i] and "%20financieros" in Links_descarga_modif[i]) 
                        ]
    #20231117estados_ financieros_ahorro_credito_sept_2023.xlsx
    links_descarga_filtrados = [list_links[i] for i in indices_a_conservar]
    return links_descarga_filtrados

def descargar_archivo(url_base, link, carpeta_destino="Data_historico"):
    url_excel_remote = url_base + link
    try:
        respuesta = requests.get(url_excel_remote, timeout=30, verify=False)
        respuesta.raise_for_status()
        nombre_archivo = nombre(url_excel_remote)
        ruta_completa = os.path.join(carpeta_destino, nombre_archivo)

        with open(ruta_completa, "wb") as f:
            f.write(respuesta.content)
        return #f"{nombre_archivo} descargado correctamente en {carpeta_destino}"
    except Exception as e:
        return f"Error descargando {url_excel_remote}: {e}"

def descargar_Archivos(lista_links, carpeta_destino="Data_historico"):
    url_base = "https://www.supersolidaria.gov.co"
    resultados = []

    # Crea la carpeta si no existe
    os.makedirs(carpeta_destino, exist_ok=True)

    with ThreadPoolExecutor(max_workers=10) as executor:
        tareas = [
            executor.submit(descargar_archivo, url_base, link, carpeta_destino)
            for link in lista_links
        ]
        for future in as_completed(tareas):
            resultados.append(future.result())

    for r in resultados:
        print(r)



In [11]:
datos_filtrados=Filtrar_datos(list_links=links_descarga)
descargar_Archivos(lista_links=datos_filtrados)

/home/lia/Descargas/Anaconda/envs/camel-model/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/lia/Descargas/Anaconda/envs/camel-model/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/home/lia/Descargas/Anaconda/envs/camel-model/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.supersolidaria.gov.co'. Adding certificate verification is strongly advised. See: https://urllib3.readthe

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
Error descargando https://www.supersolidaria.gov.co/sites/default/files/entidades/estados_financieros_ahorroycredito_noviembre2019_1.xlsx: HTTPSConnectionPool(host='www.supersolidaria.gov.co', port=443): Read timed out. (read timeout=30)
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
No

In [10]:
# Codigo que resuleve para fechas 

def sin_tildes(texto):
    return ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

def extraer_valores_head(path):
    dataframe = pd.read_excel(path, nrows=6)
    lista_columnas_1 = dataframe['Unnamed: 0'].dropna().values.flatten().tolist()
    lista_columnas_2 = dataframe['Unnamed: 1'].dropna().values.flatten().tolist()
    lista_columnas = lista_columnas_1 + lista_columnas_2
    lista_columnas = list(map(str, lista_columnas))
    lista_columnas = [col for col in lista_columnas if 'informacion' not in sin_tildes(texto=col.lower())]
    

    
    return lista_columnas


def extraer_mes(texto):
    meses_orden = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
                'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
    return [palabra for palabra in texto if palabra in meses_orden]

def mes_menor(lista_meses):
    meses_orden = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
                'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
    meses_unicos = list(set(lista_meses))
    meses_ordenados = sorted(meses_unicos, key=lambda mes: meses_orden.index(mes))
    return meses_ordenados[0] if meses_ordenados else None

def extraer_anos(texto):
    # Asegura que el texto sea string antes de usar regex
    if isinstance(texto, list):
        texto = " ".join(texto)
    return re.findall(r'\b\d{4}\b', texto)

def extraer_mes_ano(lista_valores):
    # Convertimos a minúsculas y separamos palabras
    texto_plano = [str(texto).lower().split() for texto in lista_valores if isinstance(texto, str)]

    # Extraer meses
    meses = [extraer_mes(palabras) for palabras in texto_plano]
    lista_meses = sum(meses, [])
    mes_primero = mes_menor(lista_meses)

    # Extraer años
    años = [extraer_anos(palabras) for palabras in texto_plano]
    lista_anos = sum(años, [])
    primer_ano = lista_anos[0] if lista_anos else None

    return mes_primero, primer_ano

def proceso_extraer_meses_años(path):
    lista_textos = extraer_valores_head(path)
    mes, año = extraer_mes_ano(lista_valores=lista_textos)
    return mes, año


In [11]:
def obtener_index_cuenta(data, valor="CUENTA"):
    """Busca la última fila donde aparece un valor en el DataFrame."""
    coincidencias = np.where(data == valor)[0]
    return coincidencias[-1]+1 if len(coincidencias) > 0 else 0

def leer_archivo_con_skip(path, valor="CUENTA"):
    preview = pd.read_excel(path, nrows=10)
    skip_index = obtener_index_cuenta(preview, valor)

    # Cargamos un par de filas para inspeccionar encabezados posibles
    data_pre = pd.read_excel(path, skiprows=skip_index, nrows=2)

    # Asegurar que todos los nombres de columna sean strings
    data_pre.columns = data_pre.columns.map(str)

    # Filtrar columnas que no empiezan por "Unnamed:"
    data_pre = data_pre.loc[:, ~data_pre.columns.str.startswith('Unnamed:')]

    # Verificamos si alguna columna parece ser numérica o contiene dígitos
    if any(col.strip().isdigit() or any(char.isdigit() for char in col) for col in data_pre.columns):
        row = preview.iloc[skip_index - 2].tolist()
        row = row[:2] + [x for x in row[2:] if not pd.isna(x)]
        data_lista = pd.read_excel(path, skiprows=skip_index)
        
        data_lista.columns = data_lista.columns.map(str)  # Asegura que los nombres sean strings
        data_lista = data_lista.drop(
            columns=[col for col in data_lista.columns if col.startswith("Unnamed:") and data_lista[col].isna().all()]
        )

        # Tomamos los nombres de columna reales que están al inicio
        columnas = list(data_lista.columns)[:2]
        row[:len(columnas)] = columnas
        data_lista.columns = row
        return data_lista

    return pd.read_excel(path, skiprows=skip_index)


def renombrar_columnas_duplicadas(df):
    cols = pd.Series(df.columns)
    for dup in cols[cols.duplicated()].unique():
        dups = cols[cols == dup].index.tolist()
        for i, idx in enumerate(dups[1:], start=1):
            cols[idx] = f"{dup}.repetida{i}"
    df.columns = cols
    return df


def clean_rows(data):
    data = data.copy()
    data.fillna(0, inplace=True)

    # Eliminar filas donde el valor en "CUENTA" es NaN
    data = data[data["CUENTA"].notna()]
    
    data = data.loc[
    data["NOMBRE CUENTA"].notna() & 
    (data["NOMBRE CUENTA"] != 0) & 
    (data["NOMBRE CUENTA"] != "0")]

    # Eliminar filas donde "CUENTA" no sea numérico (por ejemplo: 'abc')
    data = data[pd.to_numeric(data["CUENTA"], errors="coerce").notna()]
    

    # Renombrar columnas duplicadas (función externa)
    data = renombrar_columnas_duplicadas(data)

    return data

def read_excels(list_name_excel,carpeta):
    lista_archivos = []
    for nombre in list_name_excel:
        path = os.path.join(carpeta, nombre)
        df = leer_archivo_con_skip(path)
        mes,ano=proceso_extraer_meses_años(path)
        df["MES"]=mes
        df["AÑO"]=ano
        lista_archivos.append(df)
    return lista_archivos

In [12]:
carpeta = 'Data_historico'
nombres_archivos = [f for f in os.listdir(carpeta) if os.path.isfile(os.path.join(carpeta, f))]
#nombres_archivos

In [13]:
szs="Data_historico/f_puc_financiera_31012014_31032014_0_1.xls"
#text=extraer_valores_head(path=szs)
#extraer_mes_ano(text)
ll=["f_puc_financiera_31012014_31032014_0_1.xls"]
list_prueba=read_excels(list_name_excel=ll,
                        carpeta=carpeta)
list_prueba[0]

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE EMPLEADOS DE CAFAM LTDA,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE EMPLEADOS DE DOW COLOMBIA LTDA,...,COOPERATIVA DE AHORRO Y CREDITO FINANCIAFONDOS,COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS,COOPERATIVA DE AHORRO Y CREDITO COLANTA,MICROEMPRESAS DE ANTIOQUIA COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO CAJA UNION,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO CANAPRO,COOPERATIVA DE AHORRO Y CREDITO SUYA LTDA,MES,AÑO
0,100000,ACTIVO,81140926584.270004,8908931869.48,151057352705.970001,85061333192.460007,57085128978.279999,3050478862.35,39429150345.610001,26276461216.59,...,9868017474.66,94560900845,165570245501.429993,68067903395.870003,5236823859.2,3862933783.55,22351359898.27,22986301092.23,enero,2014
1,110000,DISPONIBLE,1852362870.78,596707614.26,337427703.91,3551153650.68,844861711.94,401257747.01,566459410.55,2146875218.32,...,574958501.79,7002726748,6158715940.69,1820673389.47,724742128.13,255911363.97,865267532.84,2674448475.36,enero,2014
2,110500,CAJA,117882589.4,3080000,333587956.59,8060266,243416650,46432740.62,76618131.43,400000,...,3546159,1000000,2814999379.93,674573770.6,72402369.55,55803296.09,215102288.36,905517804.22,enero,2014
3,110505,CAJA GENERAL,116093325.4,-,329437956.59,7460266,240336450,46432740.62,70950131.43,-,...,3112459,-,2812199379.93,669975020.6,71902369.55,54190796.09,214102288.36,901846104.22,enero,2014
4,110510,CAJA MENOR,1789264,3080000,4150000,600000,3080200,-,5668000,400000,...,433700,1000000,2800000,4598750,500000,1612500,1000000,3671700,enero,2014
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1360,935000,OTRAS ACREEDORAS DE CONTROL,-,-,-,-,-,-,250,-,...,-,-,39011209,-,-,-,-,-,enero,2014
1361,960000,ACREEDORAS CONTINGENTES POR CONTRA,133077245894.75,7498355843.02,234624671268.470001,149324333.53,53475962616.709999,2253312601,37838997417.410004,53323876614,...,7715503931,234017800553,189381837997.329987,110401215924.009995,4452600614,5578161971,18716406202,17256274065,enero,2014
1362,960500,ACREEDORAS CONTINGENTES POR CONTRA (DB),133077245894.75,7498355843.02,234624671268.470001,149324333.53,53475962616.709999,2253312601,37838997417.410004,53323876614,...,7715503931,234017800553,189381837997.329987,110401215924.009995,4452600614,5578161971,18716406202,17256274065,enero,2014
1363,980000,ACREEDORAS DE CONTROL POR CONTRA,5346883067.53,1274479801,24923367659.849998,308719488901.219971,7368750000,1170000000,6160000250,1250200000,...,1244124000,5000000000,21444096965,3696000000,2004300000,2266800000,1524265564,2947500000,enero,2014


In [14]:
list_excel=read_excels(list_name_excel=nombres_archivos,
                        carpeta=carpeta)

/home/lia/Descargas/Anaconda/envs/camel-model/lib/python3.11/site-packages/openpyxl/worksheet/header_footer.py:48: UserWarning: Cannot parse header or footer so it will be ignored
  warn("""Cannot parse header or footer so it will be ignored""")


In [112]:
"""
Ojo este texto lo escribo para mensionar que enero de 2013 precisamente es el numero 134 
y tal parece el indice esta super melo y por lo tanto cuando se limpian las lineas es donde debe salir el error, 
efectivament en la casilla de abajo estaba el error 
"""
#list_excel[121]

'\nOjo este texto lo escribo para mensionar que enero de 2013 precisamente es el numero 134 \ny tal parece el indice esta super melo y por lo tanto cuando se limpian las lineas es donde debe salir el error, \nefectivament en la casilla de abajo estaba el error \n'

In [15]:
list_excel_limpio=[]
for df in list_excel:
    if "CUENTA" in df.columns:
        list_excel_limpio.append(clean_rows(data=df))

In [16]:
Datos_2013_2025_cooperativas = pd.concat(list_excel_limpio, ignore_index=True)

# Sacar las columnas
col_mes = Datos_2013_2025_cooperativas.pop('MES')
col_ano = Datos_2013_2025_cooperativas.pop('AÑO')

# Concatenar todo junto, agregando MES y AÑO al final
Datos_2013_2025_cooperativas = pd.concat([Datos_2013_2025_cooperativas, col_mes, col_ano],axis=1)

# Renombrar columnas si es necesario
Datos_2013_2025_cooperativas.columns = list(Datos_2013_2025_cooperativas.columns[:-2]) + ['MES', 'ANO']
Datos_2013_2025_cooperativas.fillna(0, inplace=True)

In [17]:
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

# Solo columnas de texto
cols_str = Datos_2013_2025_cooperativas.select_dtypes(include=['object', 'string']).columns

# Aplica reemplazo vectorizado
Datos_2013_2025_cooperativas[cols_str] = Datos_2013_2025_cooperativas[cols_str].replace(ILLEGAL_CHARACTERS_RE, '', regex=True)


In [18]:
Datos_2013_2025_cooperativas.head()

,CUENTA,NOMBRE CUENTA,NOMBRE ENTIDAD,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,...,Unnamed: 183,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,COOPERATIVA DE AHORRO Y CREDITO COOTRAMED,COOPERATIVA DE AHORRO Y CR??DITO COMUNA,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA DEL SISTEMA NACIONAL DE JUSTICIA,COOPERATIVA DEL DEPARTAMENTO DEL CAUCA,COOPERATIVA DE TRABAJADORES DE LA CHEC LTDA..repetida1,MES,ANO
0,100000.0,ACTIVO,0,151607733771.589996,11821327551.25,389815527165.530029,163827927794.559998,118270691097.509995,11480648846.23,80435188929.039993,...,0,0,0.0,0,0,0.0,0.0,0,febrero,2024
1,110000.0,EFECTIVO Y EQUIVALENTE AL EFECTIVO,0,12131432013.290001,1042833490.5,39646323774.050003,14071347850.23,1232825237.55,2232903988.22,6840999197.81,...,0,0,0.0,0,0,0.0,0.0,0,febrero,2024
2,110500.0,CAJA,0,268998876.39,13443611.14,913797998.2,900000,416181256,181858419.93,215358874,...,0,0,0.0,0,0,0.0,0.0,0,febrero,2024
3,110505.0,CAJA GENERAL,0,259014876.39,7699511.14,908947998.2,0,410981256,181858419.93,203008874,...,0,0,0.0,0,0,0.0,0.0,0,febrero,2024
4,110510.0,CAJA MENOR,0,9984000,5744100,4850000,900000,5200000,0,12350000,...,0,0,0.0,0,0,0.0,0.0,0,febrero,2024


In [19]:
Datos_2013_2025_cooperativas_copia=Datos_2013_2025_cooperativas.copy()
len(Datos_2013_2025_cooperativas_copia)

313815

In [20]:
# Diccionario para convertir nombres de mes a número
meses_es = {
    'enero': 1, 'febrero': 2, 'marzo': 3, 'abril': 4,
    'mayo': 5, 'junio': 6, 'julio': 7, 'agosto': 8,
    'septiembre': 9, 'octubre': 10, 'noviembre': 11, 'diciembre': 12
}

# Convertir nombre del mes a número
Datos_2013_2025_cooperativas_copia['mes_num'] = Datos_2013_2025_cooperativas_copia['MES'].str.lower().map(meses_es)


Datos_2013_2025_cooperativas_copia['fecha'] = pd.to_datetime(dict(year=Datos_2013_2025_cooperativas_copia['ANO'],
                                                                month=Datos_2013_2025_cooperativas_copia['mes_num'], day=1))
Datos_2013_2025_cooperativas_copia.drop(["MES", "ANO", "mes_num"], inplace=True,axis=1)
Datos_2013_2025_cooperativas_copia.head()

,CUENTA,NOMBRE CUENTA,NOMBRE ENTIDAD,COOPERATIVA DE EMPLEADOS DE CAFAM,COOPERATIVA DE TRABAJADORES DE LA INDUSTRIA MILITAR,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,...,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO ORBISCOOP,Unnamed: 183,PRECOOPERATIVA PARA EL FOMENTO Y DESARROLLO DE LAS ACTIVIDADES MINERAS,COOPERATIVA DE AHORRO Y CREDITO COOTRAMED,COOPERATIVA DE AHORRO Y CR??DITO COMUNA,FONDO DE EMPLEADOS DEL MINISTERIO DEL INTERIOR,COOPERATIVA DEL SISTEMA NACIONAL DE JUSTICIA,COOPERATIVA DEL DEPARTAMENTO DEL CAUCA,COOPERATIVA DE TRABAJADORES DE LA CHEC LTDA..repetida1,fecha
0,100000.0,ACTIVO,0,151607733771.589996,11821327551.25,389815527165.530029,163827927794.559998,118270691097.509995,11480648846.23,80435188929.039993,...,0,0,0,0.0,0,0,0.0,0.0,0,2024-02-01
1,110000.0,EFECTIVO Y EQUIVALENTE AL EFECTIVO,0,12131432013.290001,1042833490.5,39646323774.050003,14071347850.23,1232825237.55,2232903988.22,6840999197.81,...,0,0,0,0.0,0,0,0.0,0.0,0,2024-02-01
2,110500.0,CAJA,0,268998876.39,13443611.14,913797998.2,900000,416181256,181858419.93,215358874,...,0,0,0,0.0,0,0,0.0,0.0,0,2024-02-01
3,110505.0,CAJA GENERAL,0,259014876.39,7699511.14,908947998.2,0,410981256,181858419.93,203008874,...,0,0,0,0.0,0,0,0.0,0.0,0,2024-02-01
4,110510.0,CAJA MENOR,0,9984000,5744100,4850000,900000,5200000,0,12350000,...,0,0,0,0.0,0,0,0.0,0.0,0,2024-02-01


In [21]:
#remplazar valores 
Datos_2013_2025_cooperativas_copia =  Datos_2013_2025_cooperativas_copia.replace("-", 0)
#eliminar datos que no parezcan en todos los años
anos_unicos = Datos_2013_2025_cooperativas_copia['fecha'].dt.year.unique() 
for ano in anos_unicos:
    data_ano = Datos_2013_2025_cooperativas_copia[Datos_2013_2025_cooperativas_copia['fecha'].dt.year == ano]
    columnas_datos = data_ano.drop(columns='fecha')
    columnas_a_eliminar = columnas_datos.columns[(columnas_datos == 0).all()]
    Datos_2013_2025_cooperativas_copia.drop(columns=columnas_a_eliminar, inplace=True)

In [22]:
Datos_2013_2025_cooperativas_copia

,CUENTA,NOMBRE CUENTA,COOPERATIVA DE LOS PROFESIONALES DE LA SALUD COASMEDAS,COOPERATIVA DE AHORRO Y CREDITO PARA EL BIENESTAR SOCIAL,COOPERATIVA PARA EL BIENESTAR SOCIAL,COOPERATIVA FINANCIERA SAN FRANCISCO,COOPERATIVA MULTIACTIVA DE LA AVIACION CIVIL COLOMBIANA,COOPERATIVA DE PROFESORES DE LA U NACIONAL DE COLOMBIA,CAJA COOPERATIVA CREDICOOP,COOPERATIVA DE AHORRO Y CREDITO DE SURAMERICA,...,COOPANTEX COOPERATIVA DE AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO BERLIN,COOPERATIVA DE AHORRO Y CREDITO DE AIPE,COOPERATIVA DE AHORRO Y CREDITO DE SANTANDER LIMITADA,COOPERATIVA BELEN AHORRO Y CREDITO,COOPERATIVA DE AHORRO Y CREDITO FINANCIAFONDOS,COOPERATIVA DE AHORRO Y CREDITO DE DROGUISTAS DETALLISTAS,COOPERATIVA DE AHORRO Y CREDITO COLANTA,COOPERATIVA ESPECIALIZADA DE AHORRO Y CREDITO AFROAMERICANA,fecha
0,100000.0,ACTIVO,3.898155e+11,1.638279e+11,1.182707e+11,1.148065e+10,8.043519e+10,1.557059e+11,9.147143e+10,2.257756e+10,...,3.089297e+11,4.707708e+10,3.131582e+10,1.790630e+12,3.942241e+11,1.838046e+10,2.931970e+11,3.731644e+11,7.890407e+09,2024-02-01
1,110000.0,EFECTIVO Y EQUIVALENTE AL EFECTIVO,3.964632e+10,1.407135e+10,1.232825e+09,2.232904e+09,6.840999e+09,1.182592e+10,5.680917e+09,3.345402e+09,...,3.603041e+10,4.368113e+09,1.820758e+09,2.441429e+11,2.257577e+10,1.167179e+09,6.227819e+10,3.713635e+10,9.257364e+08,2024-02-01
2,110500.0,CAJA,9.137980e+08,9.000000e+05,4.161813e+08,1.818584e+08,2.153589e+08,6.190772e+08,7.480730e+07,5.769935e+06,...,8.537958e+08,3.913779e+08,1.787160e+08,1.243465e+10,3.054532e+09,1.000000e+06,2.000000e+06,9.555685e+09,8.123824e+08,2024-02-01
3,110505.0,CAJA GENERAL,9.089480e+08,0.000000e+00,4.109813e+08,1.818584e+08,2.030089e+08,6.105772e+08,7.137156e+07,5.269935e+06,...,8.511958e+08,3.906779e+08,1.787160e+08,1.242247e+10,3.051932e+09,0.000000e+00,0.000000e+00,9.552135e+09,8.117824e+08,2024-02-01
4,110510.0,CAJA MENOR,4.850000e+06,9.000000e+05,5.200000e+06,0.000000e+00,1.235000e+07,8.500000e+06,3.435745e+06,5.000000e+05,...,2.599999e+06,7.000000e+05,0.000000e+00,1.218300e+07,2.600000e+06,1.000000e+06,2.000000e+06,3.550000e+06,6.000000e+05,2024-02-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
313810,935000.0,OTRAS ACREEDORAS DE CONTROL,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2025-12-01
313811,960000.0,ACREEDORAS POR CONTRA (DB),5.417918e+11,3.870544e+11,2.073370e+11,1.157951e+10,5.818015e+09,3.884730e+11,1.934994e+11,2.540326e+10,...,9.428047e+10,9.231831e+10,1.358747e+10,9.789575e+11,2.162036e+11,2.215258e+10,5.950021e+11,6.145372e+11,9.122729e+09,2025-12-01
313812,960500.0,RESPONSABILIDADES CONTINGENTES POR EL CONTRARIO,5.417918e+11,3.870544e+11,2.073370e+11,1.157951e+10,5.818015e+09,3.884730e+11,1.934994e+11,2.540326e+10,...,9.428047e+10,9.231831e+10,1.358747e+10,9.789575e+11,2.162036e+11,2.215258e+10,5.950021e+11,6.145372e+11,9.122729e+09,2025-12-01
313813,980000.0,ACREEDORAS DE CONTROL POR CONTRA (CR),2.713661e+10,0.000000e+00,0.000000e+00,0.000000e+00,2.135250e+10,2.213151e+10,2.700000e+10,0.000000e+00,...,6.405750e+10,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.925550e+09,0.000000e+00,0.000000e+00,0.000000e+00,2025-12-01


In [23]:
Datos_2013_2025_cooperativas_copia.to_excel("Datos_2013_2025_cooperativas.xlsx",index=False)